In [6]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, shutil, hashlib, subprocess
from pathlib import Path
DRIVE_ROOT=Path('/content/drive/MyDrive'); PARENT_DIR=DRIVE_ROOT/'CALSHIFT_Research'
PROJECT_ROOT=PARENT_DIR/'calshift-research'; CRED_DIR=DRIVE_ROOT/'.gitcreds'
subprocess.run(['git','config','--global','user.name','Md Anas Biswas'],check=False)
subprocess.run(['git','config','--global','user.email','anasbiswas@gmail.com'],check=False)
subprocess.run(['git','config','--global','credential.helper','store'],check=False)
for fn,dest in [('.git-credentials','/root/.git-credentials'),('.gitconfig','/root/.gitconfig')]:
    for cand in (PARENT_DIR/fn, CRED_DIR/fn):
        if cand.exists(): shutil.copy(cand,dest); os.chmod(dest,0o600); break
os.chdir(PROJECT_ROOT); sys.path.insert(0,str(PROJECT_ROOT/'src'))
subprocess.run(['git','pull','--ff-only','--quiet'],check=False)
import importlib
if 'config' in sys.modules: importlib.reload(sys.modules['config'])
import config
import numpy as np, pandas as pd
print('ready:', os.getcwd())


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ready: /content/drive/MyDrive/CALSHIFT_Research/calshift-research


In [2]:
# =============================================================================
# Cell 2 - rebuild source-calib-pool and target labels EXACTLY as nb15/nb16, so
# the saved probability arrays align. Randomized-APS / Mondrian / coverage fns.
# Classes: background,dos,nerisbotnet,scan11,scan44 (sorted). Focal = nerisbotnet.
# =============================================================================
UGR = config.DATASETS_DIR / 'ugr16'
src = pd.read_parquet(UGR/'july_week5.parquet'); tgt = pd.read_parquet(UGR/'august_week1.parquet')
for d in (src, tgt): d['label'] = d['label'].astype(str).str.strip().str.lower()
KEEP = ['background','dos','scan11','scan44','nerisbotnet']
src = src[src.label.isin(KEEP)].reset_index(drop=True)
tgt = tgt[tgt.label.isin(KEEP)].reset_index(drop=True)
CLASSES = sorted(KEEP); C2I = {c:i for i,c in enumerate(CLASSES)}
FOCAL = C2I['nerisbotnet']

PARTITION_SEED_UGR = 20260725
def stratified_split(df, fractions, seed, col='label'):
    rng=np.random.default_rng(seed); names=list(fractions)
    fr=np.array([fractions[k] for k in names],float); big=names[int(np.argmax(fr))]
    a=pd.Series(index=df.index,dtype=object)
    for _,s in df.groupby(col,sort=True):
        idx=s.index.to_numpy().copy(); rng.shuffle(idx); n=len(idx)
        c=np.floor(fr*n).astype(int); c[names.index(big)]+=n-c.sum(); k=0
        for nm,q in zip(names,c): a.loc[idx[k:k+q]]=nm; k+=q
    return a
src = src.assign(partition=stratified_split(src, config.SPLIT_FRACTIONS, PARTITION_SEED_UGR).values)
y_sp = src[src.partition=='source_cal_pool']['label'].map(C2I).to_numpy()   # aligns nb16 'srcpool'
y_tg = tgt['label'].map(C2I).to_numpy()                                     # aligns nb16 'target'

PROBS_DIR = config.DATA_DIR / 'ugr16_probs'
NCLS = len(CLASSES)

def dseed(*parts): return int(hashlib.sha256('|'.join(map(str,parts)).encode()).hexdigest(),16)%(2**32)
def aps_scores_all(P, rng):
    order=np.argsort(-P,axis=1); sp=np.take_along_axis(P,order,1)
    cum=np.cumsum(sp,1); U=rng.random(len(P))[:,None]
    ss=cum-(1-U)*sp; out=np.empty_like(P); np.put_along_axis(out,order,ss,1); return out
def qhat(true_scores, alpha):
    n=len(true_scores)
    if n<1: return np.inf
    return float(np.quantile(true_scores, min(np.ceil((n+1)*(1-alpha))/n,1.0), method='higher'))
def coverage(eval_scores, y_eval, cal_scores_all, y_cal, alpha):
    tc=cal_scores_all[np.arange(len(y_cal)),y_cal]
    q={c:qhat(tc[y_cal==c],alpha) for c in range(NCLS)}
    inset=np.column_stack([eval_scores[:,c]<=q[c] for c in range(NCLS)])
    cov={c:(float(inset[y_eval==c,c].mean()) if (y_eval==c).any() else np.nan) for c in range(NCLS)}
    ss=inset.sum(1)
    return cov,{c:(float(ss[y_eval==c].mean()) if (y_eval==c).any() else np.nan) for c in range(NCLS)}
print('classes:', CLASSES, '| focal idx:', FOCAL, '| src_pool:', len(y_sp), '| target:', len(y_tg))


classes: ['background', 'dos', 'nerisbotnet', 'scan11', 'scan44'] | focal idx: 2 | src_pool: 60000 | target: 400000


In [3]:
# =============================================================================
# Cell 3 - three-protocol coverage over matched draws (section 8.1). Per (seed,
# arch): R matched draws of D_eval / T_cal from the August target and S_cal from
# the July source pool. REC calibrates on D_eval, TSC on T_cal, SHC on S_cal; all
# evaluate on D_eval. Coverage per class + set size, averaged over draws.
# =============================================================================
ALPHAS=[config.ALPHA_PRIMARY]+config.ALPHA_SENSITIVITY
R=getattr(config,'N_MATCHED_DRAWS',10)
m = min(len(y_sp), len(y_tg)//2)
rows=[]
files=sorted(PROBS_DIR.glob('ugr16__*.npz'))
assert files, 'no UGR16 prob files found; run nb16 first'
for f in files:
    _,arch,seedpart = f.stem.split('__'); seed=int(seedpart.replace('seed',''))
    d=np.load(f); Psp,Ptg=d['srcpool'],d['target']
    acc={}
    for draw in range(R):
        rng=np.random.default_rng(dseed('ugr16',seed,arch,draw))
        tp=rng.permutation(len(y_tg)); de_i,tc_i=tp[:m],tp[m:2*m]
        sc_i=rng.permutation(len(y_sp))[:m]
        P_de,yde=Ptg[de_i],y_tg[de_i]; P_tc,ytc=Ptg[tc_i],y_tg[tc_i]; P_sc,ysc=Psp[sc_i],y_sp[sc_i]
        es=aps_scores_all(P_de, np.random.default_rng(dseed('ugr16',seed,arch,draw,'e')))
        cal={'REC':(aps_scores_all(P_de,np.random.default_rng(dseed('ugr16',seed,arch,draw,'r'))),yde),
             'TSC':(aps_scores_all(P_tc,np.random.default_rng(dseed('ugr16',seed,arch,draw,'t'))),ytc),
             'SHC':(aps_scores_all(P_sc,np.random.default_rng(dseed('ugr16',seed,arch,draw,'s'))),ysc)}
        for proto,(cs,yc) in cal.items():
            for a in ALPHAS:
                cov,ss=coverage(es,yde,cs,yc,a)
                for c in range(NCLS):
                    acc.setdefault((proto,c,a,'cov'),[]).append(cov[c])
                    acc.setdefault((proto,c,a,'ss'),[]).append(ss[c])
    for (proto,c,a,kind),vals in acc.items():
        if kind!='cov': continue
        rows.append({'dataset':'ugr16','environment':'july_to_august','seed':seed,'arch':arch,
                     'protocol':proto,'class':CLASSES[c],'alpha':a,
                     'coverage':round(float(np.nanmean(vals)),4),
                     'set_size':round(float(np.nanmean(acc[(proto,c,a,'ss')])),4),'nominal':round(1-a,4)})
    print('done', arch, 'seed', seed)
cov=pd.DataFrame(rows); cov.to_csv(config.REPORTS_DIR/'coverage_primary_ugr16.csv',index=False)
print('coverage rows:', len(cov))


done mlp seed 12345
done mlp seed 1337
done mlp seed 2024
done mlp seed 3407
done mlp seed 42
done mlp seed 512
done mlp seed 6021
done mlp seed 7
done mlp seed 88
done mlp seed 91
done rf seed 12345
done rf seed 1337
done rf seed 2024
done rf seed 3407
done rf seed 42
done rf seed 512
done rf seed 6021
done rf seed 7
done rf seed 88
done rf seed 91
done xgb seed 12345
done xgb seed 1337
done xgb seed 2024
done xgb seed 3407
done xgb seed 42
done xgb seed 512
done xgb seed 6021
done xgb seed 7
done xgb seed 88
done xgb seed 91
coverage rows: 1350


In [4]:
# =============================================================================
# Cell 4 - FOCAL RESULT: nerisbotnet coverage by protocol at the primary alpha
# and the TSC-vs-SHC gap. This is the criterion-1 test in a genuine fixed-support
# covariate-shift environment (S_cov 0.69). Also all-class coverage for context.
# =============================================================================
FOCAL_NAME='nerisbotnet'
prim=cov[(cov['class']==FOCAL_NAME)&(cov['alpha']==config.ALPHA_PRIMARY)]
overall=prim.groupby('protocol')['coverage'].mean().round(4)
print(f'{FOCAL_NAME} coverage at alpha={config.ALPHA_PRIMARY} (nominal {round(1-config.ALPHA_PRIMARY,3)}):')
print(overall.to_string())
gap=float(overall.get('TSC',np.nan)-overall.get('SHC',np.nan))
print(f'\nfocal gap  TSC - SHC = {gap:+.4f}')
print('SHC undercovers focal:', bool(overall.get('SHC',1.0) < (1-config.ALPHA_PRIMARY)-0.02))

print('\nall-class coverage by protocol at primary alpha:')
allc=cov[cov['alpha']==config.ALPHA_PRIMARY].groupby(['class','protocol'])['coverage'].mean().unstack('protocol').round(3)
print(allc.to_string())

# per-architecture focal, to confirm the effect is not one model
byarch=prim.groupby(['arch','protocol'])['coverage'].mean().unstack('protocol').round(4)
print('\nfocal coverage by architecture:')
print(byarch.to_string())

verdict={'dataset':'ugr16','environment':'july_to_august','focal_class':FOCAL_NAME,
         'alpha_primary':config.ALPHA_PRIMARY,'nominal':round(1-config.ALPHA_PRIMARY,4),
         'S_cov':0.6925,'shift_type':'fixed-support temporal covariate',
         'focal_coverage_by_protocol':{k:float(v) for k,v in overall.items()},
         'focal_gap_TSC_minus_SHC':round(gap,4),
         'reads':'criterion-1 test: does SHC undercover under covariate shift alone'}
(config.REPORTS_DIR/'ugr16_focal_verdict.json').write_text(json.dumps(verdict,indent=2))
print('\n', json.dumps(verdict, indent=2))


nerisbotnet coverage at alpha=0.05 (nominal 0.95):
protocol
REC    0.9499
SHC    0.9473
TSC    0.9497

focal gap  TSC - SHC = +0.0024
SHC undercovers focal: False

all-class coverage by protocol at primary alpha:
protocol       REC    SHC    TSC
class                           
background   0.950  0.949  0.950
dos          0.950  0.950  0.950
nerisbotnet  0.950  0.947  0.950
scan11       0.950  0.535  0.950
scan44       0.982  0.799  0.982

focal coverage by architecture:
protocol     REC     SHC     TSC
arch                            
mlp       0.9497  0.9432  0.9498
rf        0.9501  0.9490  0.9498
xgb       0.9498  0.9497  0.9494

 {
  "dataset": "ugr16",
  "environment": "july_to_august",
  "focal_class": "nerisbotnet",
  "alpha_primary": 0.05,
  "nominal": 0.95,
  "S_cov": 0.6925,
  "shift_type": "fixed-support temporal covariate",
  "focal_coverage_by_protocol": {
    "REC": 0.9499,
    "SHC": 0.9473,
    "TSC": 0.9497
  },
  "focal_gap_TSC_minus_SHC": 0.0024,
  "reads": "criter

In [5]:
# =============================================================================
# Cell 5 - commit.
# =============================================================================
def git(*a, show=True):
    r=subprocess.run(['git',*a],capture_output=True,text=True)
    if show and (r.stdout or r.stderr): print((r.stdout+r.stderr).strip())
    return r
for s,dd in [('/root/.git-credentials',PARENT_DIR/'.git-credentials'),('/root/.gitconfig',PARENT_DIR/'.gitconfig')]:
    if os.path.exists(s): shutil.copy(s,dd)
os.chdir(PROJECT_ROOT); git('add','-A',show=False)
if git('status','--porcelain',show=False).stdout.strip():
    git('commit','-m','nb17: UGR16 three-protocol conformal coverage, nerisbotnet focal gap (criterion-1 covariate test)')
    r=git('push','-u','origin','main')
    if r.returncode: print('PUSH FAILED. Commit is safe locally.')
else: print('nothing to commit')
print(git('log','--oneline','-3',show=False).stdout)


[main 3dce2df] nb17: UGR16 three-protocol conformal coverage, nerisbotnet focal gap (criterion-1 covariate test)
 3 files changed, 1368 insertions(+)
 create mode 100644 notebooks/17_ugr16_conformal_coverage.ipynb
 create mode 100644 reports/coverage_primary_ugr16.csv
 create mode 100644 reports/ugr16_focal_verdict.json
Branch 'main' set up to track remote branch 'main' from 'origin'.
To https://github.com/anasbiswas1/calshift-research.git
   d2fbb4f..3dce2df  main -> main
3dce2df nb17: UGR16 three-protocol conformal coverage, nerisbotnet focal gap (criterion-1 covariate test)
d2fbb4f nb16: UGR16 multiclass model panel + isotonic calibration (30/30)
99791d3 nb16: UGR16 multiclass model panel + isotonic calibration (30/30)



In [7]:
import subprocess, os
from pathlib import Path
os.chdir('/content/drive/MyDrive/CALSHIFT_Research/calshift-research')
assert Path('reports/UGR16_RESULTS.md').exists(), 'drop the file into reports/ first'
def git(*a, show=True):
    r=subprocess.run(['git',*a],capture_output=True,text=True)
    if show and (r.stdout or r.stderr): print((r.stdout+r.stderr).strip())
    return r
git('add','reports/UGR16_RESULTS.md', show=False)
git('commit','-m','results: UGR16 selective covariate-shift finding (nerisbotnet holds, scans collapse)')
git('push')
git('log','--oneline','-2')

[main 3c499f4] results: UGR16 selective covariate-shift finding (nerisbotnet holds, scans collapse)
 1 file changed, 39 insertions(+)
 create mode 100644 reports/UGR16_RESULTS.md
To https://github.com/anasbiswas1/calshift-research.git
   3dce2df..3c499f4  main -> main
3c499f4 results: UGR16 selective covariate-shift finding (nerisbotnet holds, scans collapse)
3dce2df nb17: UGR16 three-protocol conformal coverage, nerisbotnet focal gap (criterion-1 covariate test)


CompletedProcess(args=['git', 'log', '--oneline', '-2'], returncode=0, stdout='3c499f4 results: UGR16 selective covariate-shift finding (nerisbotnet holds, scans collapse)\n3dce2df nb17: UGR16 three-protocol conformal coverage, nerisbotnet focal gap (criterion-1 covariate test)\n', stderr='')